# Task 01 — Sentiment Analysis: Valutazione

Confronto tra **Gemma 3 1B base** e **Gemma 3 1B fine-tuned** su SENTIPOLC 2016.

Metriche: **Accuracy** e **macro-F1**

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

# Configurazione
CHECKPOINT_DIR = '../checkpoints/task_01_sentiment/adapter'
BASE_MODEL     = 'unsloth/gemma-3-1b-it-bnb-4bit'
DATASET_ID     = 'evalitahf/sentiment_analysis'
MAX_EVAL       = 200   # esempi di test
LABEL_MAP      = {0: 'negativo', 1: 'positivo', 2: 'neutro'}
LABELS         = ['negativo', 'positivo', 'neutro']

print('Configurazione caricata ✓')

In [ ]:
# ── Caricamento dataset di test ──────────────────────────────
ds = load_dataset(DATASET_ID, trust_remote_code=True)
test_split = ds.get('test', ds['train'].select(range(MAX_EVAL)))
if len(test_split) > MAX_EVAL:
    test_split = test_split.select(range(MAX_EVAL))

print(f'Esempi di test: {len(test_split)}')
pd.DataFrame(test_split[:5]).head()

In [ ]:
# ── Caricamento modelli ──────────────────────────────────────
from unsloth import FastLanguageModel
from peft import PeftModel

# Modello base
print('Caricamento modello BASE...')
base_model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=512, load_in_4bit=True
)
FastLanguageModel.for_inference(base_model)

# Modello fine-tuned
print('Caricamento modello FINE-TUNED...')
ft_model, _ = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=512, load_in_4bit=True
)
ft_model = PeftModel.from_pretrained(ft_model, CHECKPOINT_DIR)
FastLanguageModel.for_inference(ft_model)

print('Modelli caricati ✓')

In [ ]:
# ── Funzione di inferenza ────────────────────────────────────
SYSTEM_PROMPT = (
    'Sei un sistema di analisi del sentiment. '
    'Rispondi sempre e solo con una parola: positivo, negativo o neutro.'
)

def predict_sentiment(model, tokenizer, texts: list[str]) -> list[str]:
    """Predice il sentiment per una lista di testi."""
    preds = []
    for text in texts:
        prompt = (
            f'<start_of_turn>user\n'
            f'{SYSTEM_PROMPT}\n\n'
            f'Testo: {text}\n'
            f'<end_of_turn>\n'
            f'<start_of_turn>model\n'
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=5,
                do_sample=False, temperature=1.0, repetition_penalty=1.1
            )
        decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        # Normalizza output
        pred = decoded.strip().lower()
        if 'pos' in pred:   pred = 'positivo'
        elif 'neg' in pred: pred = 'negativo'
        else:               pred = 'neutro'
        preds.append(pred)
    return preds

print('Funzione di inferenza pronta ✓')

In [ ]:
# ── Valutazione ──────────────────────────────────────────────
texts  = [r.get('text', r.get('sentence', '')) for r in test_split]
y_true = [LABEL_MAP.get(int(r.get('label', 2)), 'neutro') for r in test_split]

print('Predizioni modello BASE...')
y_base = predict_sentiment(base_model, tokenizer, texts)

print('Predizioni modello FINE-TUNED...')
y_ft   = predict_sentiment(ft_model, tokenizer, texts)

# Metriche
acc_base = accuracy_score(y_true, y_base)
acc_ft   = accuracy_score(y_true, y_ft)
f1_base  = f1_score(y_true, y_base, average='macro', zero_division=0, labels=LABELS)
f1_ft    = f1_score(y_true, y_ft,   average='macro', zero_division=0, labels=LABELS)

print(f'\n{"":-<45}')
print(f'{"Modello":<25} {"Accuracy":>8} {"macro-F1":>10}')
print(f'{"":-<45}')
print(f'{"Base (Gemma 3 1B)":<25} {acc_base:>8.3f} {f1_base:>10.3f}')
print(f'{"Fine-tuned":<25} {acc_ft:>8.3f} {f1_ft:>10.3f}')
print(f'{"Δ (fine-tuned - base)":<25} {acc_ft-acc_base:>+8.3f} {f1_ft-f1_base:>+10.3f}')
print(f'{"":-<45}')

In [ ]:
# ── Report per classe ────────────────────────────────────────
print('=== BASE ===')
print(classification_report(y_true, y_base, labels=LABELS, zero_division=0))
print('=== FINE-TUNED ===')
print(classification_report(y_true, y_ft, labels=LABELS, zero_division=0))

In [ ]:
# ── Grafici ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy & F1 bar chart
metrics = ['Accuracy', 'macro-F1']
base_vals = [acc_base, f1_base]
ft_vals   = [acc_ft,   f1_ft]
x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, base_vals, w, label='Base', color='#4472C4')
axes[0].bar(x + w/2, ft_vals,   w, label='Fine-tuned', color='#ED7D31')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1); axes[0].set_title('Task 01 — Sentiment: Metriche')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Distribuzione predizioni
from collections import Counter
counts_ft = Counter(y_ft)
axes[1].bar(counts_ft.keys(), counts_ft.values(), color=['#FF4444','#44BB44','#AAAAAA'])
axes[1].set_title('Distribuzione predizioni (fine-tuned)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../checkpoints/task_01_sentiment/evaluation_results.png', dpi=150)
plt.show()
print('Grafico salvato ✓')

In [ ]:
# ── Esempi qualitativi ───────────────────────────────────────
N_SHOW = 10
df = pd.DataFrame({
    'Testo':        [t[:80] + '...' if len(t) > 80 else t for t in texts[:N_SHOW]],
    'Vera label':   y_true[:N_SHOW],
    'Base':         y_base[:N_SHOW],
    'Fine-tuned':   y_ft[:N_SHOW],
})
df['Base ✓'] = df['Vera label'] == df['Base']
df['FT ✓']   = df['Vera label'] == df['Fine-tuned']
print(df.to_string(index=False))